[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/39.%20The%20Channel%20Dredging%20%26%20Deepening%20Investment%20Problem/P39-Tier-5_executed.ipynb)

---


# 39. The Channel Dredging and Deepening Investment Problem

## Tier 5: Digital Twin System-of-Systems

### Goal and scope

This notebook builds a small, synthetic digital-twin-style monitoring simulation for a three-segment channel over a seven-day period. It generates noisy sensor observations, updates a virtual system state at fixed time intervals, calculates summary metrics, and visualizes the resulting time series. The example illustrates how data, a simulation model, and system-level monitoring can be organized in one workflow.

The implementation is a teaching prototype, not a deployed digital twin. It does not connect to physical sensors, forecast future conditions, optimize investment timing, learn from historical data, or simulate dredging operations during the run. The notebook defines illustrative sedimentation and dredging calculations, but they are not coupled to the monitoring simulation. Consequently, the results describe the behavior of this synthetic monitoring model, not actual channel performance or investment recommendations.

### Inputs and outputs

**Inputs** include the three channel segments and their initial and target depths, a simulation start time and duration, a fixed update interval, and assumptions for synthetic sensor readings and economic indicators.

**Outputs** include timestamped sensor records and system-state snapshots, per-step channel and traffic metrics, a compact performance analysis, visualizations, and simple integration and stress indicators.

### Model assumptions

- Each segment has a fixed initial physical depth during this simulation. Sensor depth readings contain independent measurement noise around that depth.
- The latest sensor reading is used as the digital twin's current depth estimate. It is not a simulated physical change in the channel.
- Flow, vessel counts, weather, visibility, and wave height are generated synthetically. Their distributions are illustrative rather than calibrated to a particular port.
- The economic indicator values are example inputs. The delay-cost metric multiplies the vessel count observed during an interval by a stated cost per vessel-hour; it is a simplified interval-level proxy, not a complete economic model.
- The sedimentation equation is an illustrative empirical proxy. Its coefficient absorbs scaling and is not a dimensionally calibrated sediment-transport law.
- The simulation uses stochastic sensor and operating-condition generation without a fixed random seed. Repeated executions can therefore produce different time series.
- Investment decisions, environmental impacts, vessel-routing interactions, forecast uncertainty, and model calibration are outside the implemented scope.

### Method and mathematical model

At each discrete time step, the simulation generates one sensor record for each channel segment. The noisy depth observation becomes the segment's current estimated depth, and the observed vessel count becomes its traffic value. The resulting system state is stored before time advances. This sequence is the notebook's central digital-twin workflow: generate observations, synchronize the virtual state, calculate metrics, and retain a time-indexed record.

Let $i \in \mathcal{S}$ index channel segments and $k \in \{0,\ldots,T-1\}$ index simulation steps. The observation model for depth is

$$
y_{i,k} = d_i + \varepsilon_{i,k},
\qquad
\varepsilon_{i,k} \sim \mathcal{N}(0,\sigma_d^2),
$$

where $d_i$ is the fixed physical depth specified for segment $i$, $y_{i,k}$ is its noisy sensor reading, and $\sigma_d=0.05$ meters is the standard deviation of the illustrative measurement noise. In the code, `generate_sensor_data` produces the observations; `physical_depths` stores $d_i$ and `SensorData.depth` stores $y_{i,k}$.

The virtual state estimate and traffic observation are updated as

$$
\widehat{d}_{i,k} = y_{i,k},
\qquad
q_{i,k} = \text{observed vessel count for segment } i.
$$

Here, $\widehat{d}_{i,k}$ is the latest estimated depth and $q_{i,k}$ is the traffic count. The code stores these values in `DigitalTwinState.channel_conditions` and `DigitalTwinState.traffic_patterns`. This direct update makes the observation-to-state synchronization explicit; it is not a filtering or forecasting method.

For a step of duration $\Delta t$, the principal monitoring metrics are

$$
\overline{d}_k = \frac{1}{|\mathcal{S}|}\sum_{i\in\mathcal{S}}\widehat{d}_{i,k},
\qquad
d^{\min}_k = \min_{i\in\mathcal{S}}\widehat{d}_{i,k},
$$

$$
Q_k = \sum_{i\in\mathcal{S}} q_{i,k},
\qquad
C_k = Q_k c_{\mathrm{delay}},
\qquad
\eta_k = \frac{\overline{d}_k}{d_{\mathrm{target}}}.
$$

In the implementation, these quantities are `avg_channel_depth`, `min_channel_depth`, `total_traffic`, `total_delay_cost`, and `system_efficiency`. The last quantity is a depth-to-target ratio, not a general measure of operational efficiency; it can exceed 1 if the estimated average depth is greater than the target. The delay-cost proxy uses the configured `vessel_delay_cost` as $c_{\mathrm{delay}}$ and assumes the observed vessel count represents vessels incurring one interval's delay cost.

The simulation contains $T = \text{duration}/\Delta t$ updates. Generating and processing each segment once per step takes $O(T|\mathcal{S}|)$ time. Retaining all state snapshots also requires memory proportional to the number of steps and segments, in addition to the bounded recent-sensor history. These costs are small for the example configuration; a real deployment would also need to address data volume, synchronization, validation, and model calibration.

### How to interpret the results

The plots and metrics summarize the synthetic observations and the model's state updates. Small changes in depth can arise from sensor noise, not from channel evolution. A high depth-to-target ratio indicates that the simulated estimated depth is closer to or above the configured target; it does not demonstrate that dredging has occurred. The integration and stress indicators are simple diagnostics defined below, not evidence of real-world reliability or compliance with an external standard.

In [ ]:
from copy import deepcopy
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class SensorData:
    """Synthetic sensor observations for one channel segment and time."""

    timestamp: datetime
    segment_id: int
    depth: float
    flow_rate: float
    vessel_count: int
    weather_condition: str
    visibility: float
    wave_height: float


@dataclass
class DredgingOperation:
    """State of a dredging operation represented by the model."""

    segment_id: int
    vessel_id: str
    operation_type: str
    target_depth: float
    current_depth: float
    progress: float
    estimated_completion: datetime
    cost_per_hour: float


@dataclass
class EconomicIndicator:
    """Illustrative economic inputs associated with the system state."""

    timestamp: datetime
    fuel_cost: float
    vessel_delay_cost: float
    environmental_cost: float
    market_demand: float
    investment_budget: float


@dataclass
class DigitalTwinState:
    """Virtual state, recent observations, and associated indicators."""

    current_time: datetime
    sensor_data: List[SensorData]
    dredging_operations: List[DredgingOperation]
    economic_indicators: EconomicIndicator
    channel_conditions: Dict[int, float]
    traffic_patterns: Dict[int, int]
    forecast_data: Dict[str, Any]


simulation_start = datetime(2024, 1, 1, 8, 0, 0)
simulation_duration = timedelta(days=7)
update_interval = timedelta(hours=1)

segments_config = {
    0: {
        "name": "Entrance Channel",
        "current_depth": 11.6,
        "target_depth": 14.0,
        "length": 5.2,
        "soil_type": "sand",
    },
    1: {
        "name": "Inner Channel",
        "current_depth": 11.6,
        "target_depth": 14.0,
        "length": 4.8,
        "soil_type": "mixed",
    },
    2: {
        "name": "Berth Approach",
        "current_depth": 11.6,
        "target_depth": 14.0,
        "length": 6.1,
        "soil_type": "rock",
    },
}

total_simulation_steps = int(simulation_duration / update_interval)
print("Synthetic Digital Twin Configuration")
print(f"- Simulation period: {simulation_duration.days} days")
print(f"- Update interval: {update_interval.total_seconds() / 3600:g} hours")
print(f"- Channel segments: {len(segments_config)}")
print(f"- Total simulation steps: {total_simulation_steps}")

## 1. Illustrative physics calculations

A digital twin can connect a virtual state to models of physical processes. The class below provides two small illustrative calculations: a sedimentation proxy and a dredging-progress update. They clarify how such components can be represented in code, but the seven-day monitoring loop does not call them. In particular, the channel depths in the main simulation remain fixed physically and vary only through noisy sensor observations.

The sedimentation proxy subtracts a nonnegative depth change from the current depth and prevents the computed depth from becoming negative. Its coefficient and flow scaling are demonstration assumptions, not a calibrated physical law. The dredging update treats `progress` as a fraction of the total planned depth change: the remaining depth gap and remaining progress fraction are advanced consistently, with progress capped at 1.

In [ ]:
class PhysicsBasedSimulator:
    """Illustrative calculations for channel and dredging dynamics."""

    def __init__(self, segments_config: Dict[int, Dict[str, Any]]) -> None:
        self.segments_config = segments_config
        self.sediment_transport_rate = 0.001
        self.dredging_efficiency = 0.85
        self.weather_impact_factor = 1.2

    def simulate_sedimentation(
        self,
        current_depth: float,
        flow_rate: float,
        time_hours: float,
    ) -> float:
        """Apply an illustrative, nonnegative sedimentation depth loss."""
        sedimentation = (
            self.sediment_transport_rate * flow_rate * time_hours * 0.1
        )
        return max(0.0, current_depth - sedimentation)

    def simulate_dredging_progress(
        self,
        operation: DredgingOperation,
        weather_factor: float,
    ) -> DredgingOperation:
        """Advance operation depth and progress by a bounded fraction."""
        if weather_factor <= 0:
            raise ValueError("weather_factor must be greater than zero")

        effective_efficiency = self.dredging_efficiency / weather_factor
        remaining_progress = max(0.0, 1.0 - operation.progress)
        progress_increase = min(effective_efficiency * 0.05, remaining_progress)
        depth_to_remove = max(
            0.0, operation.target_depth - operation.current_depth
        )

        if remaining_progress > 0.0:
            depth_increase = (
                depth_to_remove * progress_increase / remaining_progress
            )
        else:
            depth_increase = 0.0

        new_progress = min(1.0, operation.progress + progress_increase)
        new_depth = min(operation.target_depth, operation.current_depth + depth_increase)

        return DredgingOperation(
            segment_id=operation.segment_id,
            vessel_id=operation.vessel_id,
            operation_type=operation.operation_type,
            target_depth=operation.target_depth,
            current_depth=new_depth,
            progress=new_progress,
            estimated_completion=operation.estimated_completion,
            cost_per_hour=operation.cost_per_hour,
        )


print("Illustrative physics calculations are defined.")

## 2. Digital-twin state and update cycle

The system object brings together the segment configuration, the current virtual state, synthetic sensor generation, and per-step metrics. A time step follows a fixed order: generate observations at the current time, update the state estimate, save a snapshot, calculate metrics, and then advance the clock. Saving a deep copy matters because later updates mutate the live state; without snapshots, earlier simulation records would inadvertently refer to the final state.

The constructor receives the update interval explicitly so the clock advances according to the simulation configuration. The state includes fields for operations and forecasts to represent possible system components, but both remain empty in this monitoring-only example.

In [ ]:
class DigitalTwinSystem:
    """Synthetic monitoring simulation for a multi-segment channel."""

    def __init__(
        self,
        segments_config: Dict[int, Dict[str, Any]],
        simulation_start: datetime,
        simulation_duration: timedelta,
        update_interval: timedelta,
    ) -> None:
        if update_interval.total_seconds() <= 0:
            raise ValueError("update_interval must be greater than zero")

        self.segments_config = segments_config
        self.simulation_start = simulation_start
        self.simulation_duration = simulation_duration
        self.update_interval = update_interval
        self.current_time = simulation_start
        self.physics_simulator = PhysicsBasedSimulator(segments_config)

        self.physical_depths = {
            segment_id: config["current_depth"]
            for segment_id, config in segments_config.items()
        }

        self.state = DigitalTwinState(
            current_time=simulation_start,
            sensor_data=[],
            dredging_operations=[],
            economic_indicators=EconomicIndicator(
                timestamp=simulation_start,
                fuel_cost=500.0,
                vessel_delay_cost=1000.0,
                environmental_cost=5000.0,
                market_demand=1.0,
                investment_budget=50.0,
            ),
            channel_conditions=dict(self.physical_depths),
            traffic_patterns={segment_id: 0 for segment_id in segments_config},
            forecast_data={},
        )

        self.historical_states: List[DigitalTwinState] = []
        self.decision_history: List[Dict[str, Any]] = []
        self.performance_metrics: List[Dict[str, float]] = []

    def generate_sensor_data(self, timestamp: datetime) -> List[SensorData]:
        """Generate synthetic sensor observations for every segment."""
        sensor_data = []

        for segment_id in self.segments_config:
            base_depth = self.physical_depths[segment_id]
            depth_variation = np.random.normal(0.0, 0.05)

            hour_of_day = timestamp.hour
            flow_rate = (
                10.0
                + 5.0 * np.sin(2.0 * np.pi * hour_of_day / 24.0)
                + np.random.normal(0.0, 1.0)
            )

            vessel_count = int(
                8.0
                + 4.0 * np.sin(2.0 * np.pi * hour_of_day / 24.0 - np.pi / 2.0)
                + np.random.normal(0.0, 1.0)
            )
            vessel_count = max(0, vessel_count)

            weather_conditions = ["clear", "cloudy", "rainy", "foggy"]
            weather_probabilities = [0.4, 0.3, 0.2, 0.1]
            weather = np.random.choice(
                weather_conditions, p=weather_probabilities
            )

            visibility_ranges = {
                "clear": (8.0, 10.0),
                "cloudy": (5.0, 8.0),
                "rainy": (2.0, 5.0),
                "foggy": (0.5, 2.0),
            }
            visibility = np.random.uniform(*visibility_ranges[weather])

            if weather == "clear":
                wave_height = np.random.uniform(0.1, 0.8)
            else:
                wave_height = np.random.uniform(0.5, 2.0)

            sensor_data.append(
                SensorData(
                    timestamp=timestamp,
                    segment_id=segment_id,
                    depth=base_depth + depth_variation,
                    flow_rate=max(0.0, flow_rate),
                    vessel_count=vessel_count,
                    weather_condition=str(weather),
                    visibility=visibility,
                    wave_height=wave_height,
                )
            )

        return sensor_data

    def simulate_step(self) -> Dict[str, Any]:
        """Update, snapshot, measure, and advance one simulation interval."""
        self.update_system_state(self.current_time)

        state_snapshot = deepcopy(self.state)
        self.historical_states.append(state_snapshot)

        metrics = self._calculate_performance_metrics()
        self.performance_metrics.append(metrics)

        result = {
            "timestamp": self.state.current_time,
            "system_state": state_snapshot,
            "performance_metrics": metrics.copy(),
        }
        self.current_time += self.update_interval
        return result

    def update_system_state(self, timestamp: datetime) -> None:
        """Synchronize the virtual state with the latest observations."""
        new_sensor_data = self.generate_sensor_data(timestamp)

        for sensor in new_sensor_data:
            self.state.channel_conditions[sensor.segment_id] = sensor.depth
            self.state.traffic_patterns[sensor.segment_id] = sensor.vessel_count

        self.state.sensor_data.extend(new_sensor_data)
        cutoff_time = timestamp - timedelta(hours=24)
        self.state.sensor_data = [
            sensor
            for sensor in self.state.sensor_data
            if sensor.timestamp >= cutoff_time
        ]
        self.state.current_time = timestamp

    def _calculate_performance_metrics(self) -> Dict[str, float]:
        """Calculate depth, traffic, cost-proxy, and target-ratio metrics."""
        depths = list(self.state.channel_conditions.values())
        average_depth = float(np.mean(depths))
        minimum_depth = float(min(depths))
        total_traffic = int(sum(self.state.traffic_patterns.values()))
        delay_cost = (
            total_traffic * self.state.economic_indicators.vessel_delay_cost
        )
        target_depth = float(
            np.mean(
                [config["target_depth"] for config in self.segments_config.values()]
            )
        )

        return {
            "avg_channel_depth": average_depth,
            "min_channel_depth": minimum_depth,
            "total_traffic": float(total_traffic),
            "total_delay_cost": float(delay_cost),
            "active_operations": float(len(self.state.dredging_operations)),
            "system_efficiency": average_depth / target_depth,
        }


print("Digital twin monitoring system is defined.")

## 3. Run the seven-day simulation

The loop runs once per configured update interval. Each returned record contains the timestamp, a stable snapshot of the virtual state, and the metrics for that step. Progress messages are informational; the final segment values are the last noisy sensor-based estimates, not depths produced by a dredging process.

In [ ]:
digital_twin = DigitalTwinSystem(
    segments_config=segments_config,
    simulation_start=simulation_start,
    simulation_duration=simulation_duration,
    update_interval=update_interval,
)

print("=" * 70)
print("SYNTHETIC CHANNEL MONITORING SIMULATION")
print("=" * 70)
print(f"- Simulation start: {simulation_start}")
print(f"- Duration: {simulation_duration}")
print(f"- Update interval: {update_interval}")
print(f"- Planned time steps: {total_simulation_steps}")

simulation_results = []
for step in range(total_simulation_steps):
    step_result = digital_twin.simulate_step()
    simulation_results.append(step_result)

    if step % 12 == 0:
        metrics = step_result["performance_metrics"]
        print(
            f"Step {step:3d}: {step_result['timestamp']:%Y-%m-%d %H:%M} - "
            f"Average estimated depth: {metrics['avg_channel_depth']:.2f} m, "
            f"observed traffic: {metrics['total_traffic']:.0f}, "
            f"active operations: {metrics['active_operations']:.0f}"
        )

assert len(simulation_results) == total_simulation_steps
assert len(digital_twin.historical_states) == total_simulation_steps

print(f"\nCompleted {len(simulation_results)} simulation steps.")
final_state = simulation_results[-1]["system_state"]

print("\nFinal sensor-based state estimates")
for segment_id, depth in final_state.channel_conditions.items():
    segment = segments_config[segment_id]
    traffic = final_state.traffic_patterns[segment_id]
    deficit = max(0.0, segment["target_depth"] - depth)
    print(
        f"- {segment['name']}: estimated depth {depth:.2f} m; "
        f"target {segment['target_depth']:.2f} m; "
        f"target gap {deficit:.2f} m; observed traffic {traffic}"
    )

## 4. Analyze the simulation time series

Analysis separates extraction of time-indexed values from their summary. Keeping this step explicit makes it easier to check which data contribute to each reported quantity. The total simulated time is calculated from the number of observations and the configured interval, so the summary remains meaningful if the interval is changed.

In [ ]:
def analyze_digital_twin_performance(
    results: List[Dict[str, Any]],
    update_interval: timedelta,
) -> Dict[str, Any]:
    """Extract time series used in the monitoring summary and plots."""
    if not results:
        raise ValueError("results must contain at least one simulation step")
    if update_interval.total_seconds() <= 0:
        raise ValueError("update_interval must be greater than zero")

    timestamps = [result["timestamp"] for result in results]
    metrics = [result["performance_metrics"] for result in results]

    return {
        "timestamps": timestamps,
        "avg_depths": [item["avg_channel_depth"] for item in metrics],
        "min_depths": [item["min_channel_depth"] for item in metrics],
        "system_efficiency": [item["system_efficiency"] for item in metrics],
        "total_traffic": [item["total_traffic"] for item in metrics],
        "delay_costs": [item["total_delay_cost"] for item in metrics],
        "total_simulation_hours": (
            len(results) * update_interval.total_seconds() / 3600.0
        ),
    }


performance_analysis = analyze_digital_twin_performance(
    simulation_results,
    update_interval,
)

print("\n" + "=" * 60)
print("DIGITAL TWIN MONITORING SUMMARY")
print("=" * 60)
print(
    f"- Simulated interval coverage: "
    f"{performance_analysis['total_simulation_hours']:.1f} hours"
)
print(
    f"- Mean estimated channel depth: "
    f"{np.mean(performance_analysis['avg_depths']):.2f} m"
)
print(
    f"- Lowest estimated segment depth: "
    f"{np.min(performance_analysis['min_depths']):.2f} m"
)
print(
    f"- Mean depth-to-target ratio: "
    f"{np.mean(performance_analysis['system_efficiency']) * 100:.1f}%"
)
print(
    f"- Sum of observed segment traffic across steps: "
    f"{np.sum(performance_analysis['total_traffic']):.0f}"
)
print(
    f"- Mean interval-level delay-cost proxy: "
    f"${np.mean(performance_analysis['delay_costs']):,.0f}"
)

## 5. Visualize channel, traffic, and cost-proxy behavior

The four panels show estimated depth over time, the depth-to-target ratio, traffic alongside its simplified cost proxy, and the final per-segment estimates compared with targets. The dual vertical axes in the traffic panel use different units and are labeled separately. Because the simulation does not update physical depth through dredging or sedimentation, the depth plot primarily reflects the synthetic measurement noise.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "Synthetic Channel Monitoring: Depth, Traffic, and Cost Proxy",
    fontsize=15,
    fontweight="bold",
)

hours = [
    (timestamp - simulation_start).total_seconds() / 3600.0
    for timestamp in performance_analysis["timestamps"]
]

ax1 = axes[0, 0]
ax1.plot(
    hours,
    performance_analysis["avg_depths"],
    linewidth=1.8,
    label="Mean estimated depth",
)
ax1.plot(
    hours,
    performance_analysis["min_depths"],
    linewidth=1.5,
    label="Minimum estimated depth",
)
mean_target_depth = np.mean(
    [config["target_depth"] for config in segments_config.values()]
)
mean_initial_depth = np.mean(
    [config["current_depth"] for config in segments_config.values()]
)
ax1.axhline(
    y=mean_target_depth,
    color="green",
    linestyle="--",
    label="Mean target depth",
)
ax1.axhline(
    y=mean_initial_depth,
    color="gray",
    linestyle=":",
    label="Mean configured initial depth",
)
ax1.set_xlabel("Elapsed simulation time (hours)")
ax1.set_ylabel("Depth (m)")
ax1.set_title("Estimated Channel Depth")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
depth_ratio_percent = [
    ratio * 100.0 for ratio in performance_analysis["system_efficiency"]
]
ax2.plot(
    hours,
    depth_ratio_percent,
    color="green",
    linewidth=1.8,
    label="Mean depth-to-target ratio",
)
ax2.fill_between(hours, depth_ratio_percent, alpha=0.2, color="green")
ax2.set_xlabel("Elapsed simulation time (hours)")
ax2.set_ylabel("Depth-to-target ratio (%)")
ax2.set_title("Depth Relative to the Target")
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
ax3_twin = ax3.twinx()
ax3.plot(
    hours,
    performance_analysis["total_traffic"],
    color="darkorange",
    linewidth=1.8,
    label="Observed vessel count",
)
ax3_twin.plot(
    hours,
    [cost / 1000.0 for cost in performance_analysis["delay_costs"]],
    color="red",
    linewidth=1.5,
    label="Interval cost proxy",
)
ax3.set_xlabel("Elapsed simulation time (hours)")
ax3.set_ylabel("Observed vessel count (vessels)", color="darkorange")
ax3_twin.set_ylabel("Delay-cost proxy (thousands of USD)", color="red")
ax3.set_title("Traffic and Simplified Delay-Cost Proxy")
ax3.tick_params(axis="y", labelcolor="darkorange")
ax3_twin.tick_params(axis="y", labelcolor="red")
lines, labels = ax3.get_legend_handles_labels()
twin_lines, twin_labels = ax3_twin.get_legend_handles_labels()
ax3.legend(lines + twin_lines, labels + twin_labels, loc="upper left")
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 1]
segment_ids = list(segments_config)
segment_names = [segments_config[sid]["name"] for sid in segment_ids]
final_depths = [
    final_state.channel_conditions[segment_id] for segment_id in segment_ids
]
target_depths = [
    segments_config[segment_id]["target_depth"] for segment_id in segment_ids
]
x_positions = np.arange(len(segment_names))
bar_width = 0.36
current_bars = ax4.bar(
    x_positions - bar_width / 2,
    final_depths,
    bar_width,
    label="Final estimated depth",
    color="lightblue",
    edgecolor="black",
)
target_bars = ax4.bar(
    x_positions + bar_width / 2,
    target_depths,
    bar_width,
    label="Target depth",
    color="lightgreen",
    edgecolor="black",
)
ax4.set_xlabel("Channel segment")
ax4.set_ylabel("Depth (m)")
ax4.set_title("Final Estimates and Segment Targets")
ax4.set_xticks(x_positions)
ax4.set_xticklabels(segment_names, rotation=25, ha="right")
ax4.legend()
ax4.grid(True, axis="y", alpha=0.3)

for bars in (current_bars, target_bars):
    for bar in bars:
        height = bar.get_height()
        ax4.annotate(
            f"{height:.2f}",
            (bar.get_x() + bar.get_width() / 2.0, height),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
        )

fig.tight_layout()
plt.show()
print("The plots summarize synthetic observations and model-defined indicators.")

## 6. Integration and simple resilience diagnostics

The following diagnostics quantify variation in the depth-to-target ratio and the fraction of steps that pass a model-defined threshold. They are intentionally interpreted as simple indicators, not validated measures of system stability or availability.

The stability index is $1 - s_{\eta}/\overline{\eta}$, where $s_{\eta}$ and $\overline{\eta}$ are the population standard deviation and mean of the depth-to-target ratio. It can be negative if variation exceeds the mean, so it is not intrinsically a percentage bounded between 0 and 100. The availability proxy counts a step as successful when its ratio exceeds 0.5. This threshold is a demonstration rule, not an operational service-level requirement.

A stress step is defined here as one with traffic above its 80th percentile and minimum estimated depth below its 20th percentile. These relative thresholds identify unusual observations within this run; they do not establish an external safety or stress standard.

In [ ]:
print("\n" + "=" * 60)
print("SYSTEM INTEGRATION AND RESILIENCE DIAGNOSTICS")
print("=" * 60)

efficiency_values = np.asarray(
    [result["performance_metrics"]["system_efficiency"] for result in simulation_results],
    dtype=float,
)
mean_efficiency = float(np.mean(efficiency_values))
efficiency_std = float(np.std(efficiency_values))
efficiency_stability = (
    1.0 - efficiency_std / mean_efficiency
    if not np.isclose(mean_efficiency, 0.0)
    else float("nan")
)

total_steps = len(simulation_results)
successful_steps = sum(value > 0.5 for value in efficiency_values)
availability_proxy = successful_steps / total_steps if total_steps else float("nan")

traffic_values = np.asarray(performance_analysis["total_traffic"], dtype=float)
minimum_depth_values = np.asarray(performance_analysis["min_depths"], dtype=float)
traffic_threshold = np.percentile(traffic_values, 80)
depth_threshold = np.percentile(minimum_depth_values, 20)

stress_periods = [
    index
    for index, (traffic, minimum_depth) in enumerate(
        zip(traffic_values, minimum_depth_values)
    )
    if traffic > traffic_threshold and minimum_depth < depth_threshold
]
stress_set = set(stress_periods)
normal_periods = [
    index for index in range(total_steps) if index not in stress_set
]

print("Diagnostic indicators:")
print(f"- Depth-ratio stability index: {efficiency_stability * 100:.1f}%")
print(f"- Threshold-based availability proxy: {availability_proxy * 100:.1f}%")
print(f"- Mean depth-to-target ratio: {mean_efficiency * 100:.1f}%")
print(f"- Depth-ratio standard deviation: {efficiency_std * 100:.2f} percentage points")
print(f"- Stress periods under the stated percentile rule: {len(stress_periods)}")

if stress_periods and normal_periods:
    stress_efficiency = float(np.mean(efficiency_values[stress_periods]))
    normal_efficiency = float(np.mean(efficiency_values[normal_periods]))
    degradation = (
        (1.0 - stress_efficiency / normal_efficiency) * 100.0
        if not np.isclose(normal_efficiency, 0.0)
        else float("nan")
    )
    print(f"- Mean ratio during stress periods: {stress_efficiency * 100:.1f}%")
    print(f"- Mean ratio during other periods: {normal_efficiency * 100:.1f}%")
    print(f"- Relative ratio change under stress: {degradation:.1f}%")
elif stress_periods:
    print("- All observations meet the stress rule; no comparison group is available.")
else:
    print("- No observations meet both percentile conditions.")

print(
    "These indicators describe this synthetic run only; they do not establish "
    "real-world system reliability or resilience."
)

## Summary and conclusions

This notebook demonstrates a basic system-of-systems structure for channel monitoring: a configuration of multiple segments, synthetic observations, a shared virtual state, a repeated update cycle, metric calculation, and visualization. Its key computational pattern is the explicit sequence of observation generation, state synchronization, snapshot storage, and per-step analysis.

The implemented model is deliberately narrower than a full digital twin. The physics calculations are illustrative and are not invoked by the seven-day run; no dredging operation, sediment-driven physical evolution, forecast, investment optimizer, or learning mechanism is executed. The state therefore represents noisy sensor-based estimates of fixed configured depths, alongside synthetic traffic and environmental observations. The cost and resilience quantities are simplified diagnostics whose assumptions are stated in the relevant sections.

Within these limits, the notebook provides a foundation for understanding how a monitoring simulation can organize heterogeneous inputs and produce time-indexed outputs. Any use for operational decisions would require validated physical and economic models, real data integration, calibrated uncertainty, appropriate decision logic, and independent testing. Those capabilities are outside the scope of this implementation.